# TinyOlap OLAP Lab Script Generator
This notebook converts the TinyOlap lab manual into a Python script-based implementation. It captures the manual requirements, defines the script structure, implements the code, tests sample outputs, and documents how the generated script maps to the lab instructions.

## Section 1: Review Manual Requirements
The manual describes a TinyOlap OLAP lab with the following key points:

- Install and use `tinyolap`
- Create a database, dimensions, and a cube
- Populate the cube with random sales data for metrics, years, regions, and products
- Generate OLAP views: slice, dice, pivot, drill-up, drill-down, rotation, and trend analysis
- Provide exercises to explore different OLAP operations and perspectives

The target functionality is to build a self-contained script that constructs the cube and demonstrates these OLAP operations via console output.

## Section 2: Define Script Specifications
The script should include:

- Inputs:
  - Optional seed for reproducible random data
  - A predefined cube schema and data generation logic
- Outputs:
  - Console renderings of OLAP operations
  - Reports for slice, dice, pivot, drill-up/drill-down, rotation, and trends
- Functions:
  - `create_database_and_cube`
  - `populate_cube`
  - `build_report`
  - `run_all_reports`
  - `main`
- Flow control:
  - Setup dependencies and cube schema
  - Populate data
  - Execute each OLAP operation sequentially
  - Print output for manual validation and demonstration purposes

In [ ]:
# Install TinyOlap into the active notebook kernel when it is missing.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("tinyolap") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tinyolap==0.8.27"])

from tinyolap.database import Database
from tinyolap.slice import Slice
import random


def create_database_and_cube(db_name="edu_cube"):
    db = Database(db_name)
    db.add_dimension("metrics").edit().add_many(["Actual", "Plan", "Deviation"]).commit()
    db.add_dimension("years").edit().add_many(["2022", "2023"]).commit()
    db.add_dimension("regions").edit().add_many(["North", "South"]).commit()

    editor = db.add_dimension("products").edit()
    editor.add_many(["Model X", "Model Y"])
    editor.add_many("All Products", ["Model X", "Model Y"])
    editor.commit()

    cube = db.add_cube("sales", ["metrics", "years", "regions", "products"])
    return db, cube


def populate_cube(cube, seed=42):
    random.seed(seed)
    for year in ["2022", "2023"]:
        for region in ["North", "South"]:
            for product in ["Model X", "Model Y"]:
                actual = random.randint(1000, 2000)
                plan = random.randint(800, 1900)
                deviation = actual - plan
                cube["Actual", year, region, product] = actual
                cube["Plan", year, region, product] = plan
                cube["Deviation", year, region, product] = deviation


def build_report(cube, title, header, columns, rows):
    report = Slice(cube, {
        "title": title,
        "header": header,
        "columns": columns,
        "rows": rows,
    })
    print(report.as_console_output())
    return report


def run_all_reports(cube):
    build_report(
        cube,
        title="Slice: Metrics for 2023",
        header=[{"dimension": "metrics"}],
        columns=[{"dimension": "years", "member": "2023"}],
        rows=[{"dimension": "regions"}, {"dimension": "products"}],
    )

    build_report(
        cube,
        title="Dice: 2022 North Model X",
        header=[{"dimension": "metrics"}],
        columns=[{"dimension": "years", "member": "2022"}],
        rows=[
            {"dimension": "regions", "member": "North"},
            {"dimension": "products", "member": "Model X"},
        ],
    )

    build_report(
        cube,
        title="Pivot: Products in Columns",
        header=[{"dimension": "metrics"}],
        columns=[{"dimension": "products"}],
        rows=[{"dimension": "regions"}, {"dimension": "years"}],
    )

    build_report(
        cube,
        title="Drill-Up: Aggregate - All Products",
        header=[{"dimension": "metrics"}],
        columns=[{"dimension": "years"}],
        rows=[{"dimension": "regions"}, {"dimension": "products", "member": "All Products"}],
    )

    build_report(
        cube,
        title="Drill-Down: Product Breakdown",
        header=[{"dimension": "metrics"}],
        columns=[{"dimension": "years"}],
        rows=[{"dimension": "regions"}, {"dimension": "products"}],
    )

    build_report(
        cube,
        title="Cube Rotation: Regions in Columns",
        header=[{"dimension": "metrics"}],
        columns=[{"dimension": "regions"}],
        rows=[{"dimension": "years"}, {"dimension": "products"}],
    )

    build_report(
        cube,
        title="Time Series: Product Performance",
        header=[{"dimension": "metrics"}],
        columns=[{"dimension": "years"}],
        rows=[{"dimension": "products"}],
    )


def main():
    db, cube = create_database_and_cube()
    populate_cube(cube)
    run_all_reports(cube)


if __name__ == "__main__":
    main()


## Section 3: Implement Code from Manual
The following code implements the lab steps and encapsulates them in functions.

## Section 4: Test Generated Script
This section executes the generated script logic in the notebook environment and verifies the console output. It ensures the OLAP operations defined in the manual are actually produced by the script.

In [ ]:
# Running the generated script logic here to verify the outputs.

db, cube = create_database_and_cube()
populate_cube(cube, seed=42)
run_all_reports(cube)

## Section 5: Document the Generated Script
The generated script maps directly to the manual as follows:

- `create_database_and_cube`: Implements the OLAP cube setup, dimensions, and product hierarchy.
- `populate_cube`: Adds random values for Actual, Plan, and Deviation, matching the manual's data population logic.
- `build_report`: Wraps the TinyOlap `Slice` creation and console output rendering.
- `run_all_reports`: Recreates the manual’s examples: slice, dice, pivot, drill-up, drill-down, rotation, and time series analysis.
- `main`: Provides the standard Python script entry point for running the full lab demonstration.

Use this notebook to adapt the script into a standalone `.py` file or to extend the exercises with additional OLAP views.